In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-MiniLM-L3-v2', device = "mps")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 11861.11it/s]


In [25]:
import pandas as pd

df = pd.read_parquet("hf://datasets/sentence-transformers/sentence-compression/pair/train-00000-of-00001.parquet", engine='fastparquet')

In [26]:
df_smaller = df.drop(df.sample(n=130_000).index)

In [27]:
df_smaller.head()

,text,simplified
2,It's fresh cherry time in Michigan and the bes...,It's cherry time
6,The government is defending the latest police ...,Government defends crime statistics
9,"Brightpoint ,a provider of logistic services t...",Brightpoint starts operations on Turkish market
10,A WOMAN has been seriously injured in a collis...,Woman seriously injured in collision with poli...
11,Alfalfa sprouts recalled because of salmonella...,Alfalfa sprouts recall


In [28]:
all_sources = []
all_targets = []
all_contexts = []

for full, simp in zip(df_smaller["text"].to_list(), df_smaller["simplified"].to_list()):
    hidden = model.encode(simp)

    # Concatenate along feature dim: (batch_size, 2 * hidden_size)
    # To undo, h_final = context_vector[:, :128], c_final = context_vector[:, 128:]

    all_sources.append(simp)
    all_targets.append(full)
    all_contexts.append(hidden)

context_df = pd.DataFrame({
    "source": all_sources,
    "context_vector": all_contexts,
    "target": all_targets
})

In [29]:
context_df.head()

,source,context_vector,target
0,It's cherry time,"[0.20639583, 0.26958498, 0.03340559, -0.214862...",It's fresh cherry time in Michigan and the bes...
1,Government defends crime statistics,"[-0.14154394, 0.1366636, -0.6568146, -0.102596...",The government is defending the latest police ...
2,Brightpoint starts operations on Turkish market,"[-0.22917601, -0.14567834, -0.5782237, 0.25152...","Brightpoint ,a provider of logistic services t..."
3,Woman seriously injured in collision with poli...,"[-0.029204637, 0.22790985, -0.3351024, -0.3687...",A WOMAN has been seriously injured in a collis...
4,Alfalfa sprouts recall,"[-0.39512956, 0.22274385, -0.0804363, 0.247126...",Alfalfa sprouts recalled because of salmonella...


In [30]:
context_df.to_csv("paraphrase_encodings.csv", index=False)